# Prior-Fisher drift-budget steering

`FisherDrift` treats the incoming model as the reference parameter point. It estimates a true diagonal Fisher over reference next-token contexts, computes a supervised target gradient, and applies one LoRA update with a configured quadratic drift budget. The measured KL can differ from this local approximation.

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer

from steerability.algorithms.core.steering_pipeline import SteeringPipeline
from steerability.algorithms.structural_control.fisher_drift import FisherDrift

We use a tiny model so the example runs on CPU. Reference rows describe text whose predictive behavior should be preserved. Target rows describe the behavior to learn. Production instruction-tuning data should set unwanted prompt positions in `labels` to `-100`.

In [2]:
MODEL = "hf-internal-testing/tiny-random-LlamaForCausalLM"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenized_rows(texts):
    rows = []
    for text in texts:
        row = tokenizer(text, truncation=True, max_length=64)
        row["labels"] = list(row["input_ids"])
        rows.append(row)
    return Dataset.from_list(rows)

prior_data = tokenized_rows([
    "Paris is the capital of France.",
    "Water freezes at zero degrees Celsius.",
])
target_data = tokenized_rows([
    "Question: What is 2 + 2? Answer: 4",
    "Question: What is 3 + 3? Answer: 6",
])

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.


In [ ]:
control = FisherDrift(
    prior_dataset=prior_data,
    train_dataset=target_data,
    kappa=1e-4,
    damping=1e-5,
    fisher_num_samples=2,
    r=4,
    output_dir="runs/fisher-drift-adapter",
)
pipeline = SteeringPipeline(
    model_name_or_path=MODEL,
    controls=[control],
    device="cpu",
)
pipeline.steer()
control.diagnostics

[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/21 [00:00<?, ?it/s]

In [ ]:
pipeline.generate(text="Question: What is 4 + 4? Answer:", max_new_tokens=12, do_sample=False)

For an experiment, sweep `kappa`, `damping`, and `fisher_num_samples`. Compare target-task improvement with a preservation suite and a matched-norm Euclidean update; `predicted_drift` should equal `kappa` when the target gradient is nonzero, while measured reference KL remains an empirical quantity.